In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Vivek_Vihar, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,198.91,283.18,14.14,33.86,29.50,1.12,7.85,1.87,9.84,74.62,1.74,308.41,77.36,986.29,14.08,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,223.85,319.33,16.89,35.59,32.66,1.73,7.15,2.05,15.55,77.72,1.67,293.52,67.60,986.12,14.15,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,363.65,523.48,49.19,46.95,64.96,3.36,4.99,4.45,36.59,82.65,0.88,268.67,67.02,986.19,15.01,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,396.27,542.81,51.06,66.66,71.93,2.72,4.13,5.54,52.04,82.72,2.01,205.49,69.33,986.07,14.87,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,205.69,291.01,24.51,46.69,44.77,2.52,4.83,1.81,14.48,78.30,2.15,167.02,83.71,986.00,14.72,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,356.23,510.07,23.69,72.81,57.99,2.32,13.01,3.48,18.04,53.99,0.79,236.15,89.11,977.75,18.69,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,329.19,474.40,15.20,75.18,52.26,1.32,9.94,4.04,14.58,57.77,0.64,228.40,84.11,978.14,18.27,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,268.84,421.35,13.00,73.26,49.54,1.85,12.99,4.05,13.37,55.20,0.66,245.05,85.20,977.77,18.42,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,284.67,437.28,31.36,71.39,63.38,1.86,11.34,3.58,11.27,57.19,0.64,267.58,87.25,978.14,18.10,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date   PM2.5    PM10      NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  198.91  283.18  14.140  33.86  29.500   
1  02-01-2025 00:00  03-01-2025 00:00   60.92  319.33  16.890  35.59  32.660   
2  03-01-2025 00:00  04-01-2025 00:00   60.92  161.07  10.255  46.95  23.085   
3  04-01-2025 00:00  05-01-2025 00:00   60.92  161.07  10.255  66.66  23.085   
4  05-01-2025 00:00  06-01-2025 00:00  205.69  291.01  24.510  46.69  44.770   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.12   7.85     1.87     9.84  74.62  1.74  308.41  77.36  986.29  14.08   
1  1.73   7.15     2.05    15.55  77.72  1.67  293.52  67.60  986.12  14.15   
2  0.95   4.99     4.45    36.59  82.65  0.88  268.67  67.02  986.19  15.01   
3  0.95   4.13     5.54    52.04  82.72  2.01  205.49  69.33  986.07  14.87   
4  0.95   4.83     1.81    14.48  78.30  2.15  167.02  83.71  986.00  14.72   

    RF  TOT-RF  
0  0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.747382,1.131655,0.825141,-0.061537,0.343123,0.189183,-0.940696,-0.156577,-1.243221,1.210854,-0.209051,1.103989,-1.230527,0.685363,-2.313722,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.185030,1.513321,1.322265,0.044670,0.625308,1.544356,-0.970960,-0.047311,-0.904219,1.455424,-0.297221,0.863834,-1.471048,0.648282,-2.301199,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.185030,-0.157563,0.122840,0.742070,-0.229732,-0.188488,-1.064346,1.409569,0.344922,1.844369,-1.292289,0.463038,-1.485341,0.663550,-2.147343,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.185030,-0.157563,0.122840,1.952083,-0.229732,-0.188488,-1.101528,2.071235,1.262186,1.849892,0.131036,-0.555966,-1.428415,0.637376,-2.172390,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.891463,1.214323,2.699751,0.726108,1.706722,-0.188488,-1.071264,-0.192999,-0.967745,1.501182,0.307377,-1.176433,-1.074040,0.622107,-2.199225,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.185030,-0.157563,2.551518,2.329637,2.887258,2.855097,-0.717606,0.820747,-0.756388,-0.416721,-1.405651,-0.061464,-0.940965,-1.177398,-1.488984,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.185030,-0.157563,1.016760,2.475133,2.375574,0.633502,-0.850336,1.160685,-0.961808,-0.118503,-1.594588,-0.186460,-1.064183,-1.092330,-1.564123,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.185030,2.590432,0.619061,2.357263,2.132680,1.810947,-0.718471,1.166756,-1.033645,-0.321260,-1.569397,0.082081,-1.037321,-1.173035,-1.537288,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.185030,2.758618,0.122840,2.242462,-0.229732,1.833163,-0.789808,0.881450,-1.158322,-0.164261,-1.594588,0.445458,-0.986802,-1.092330,-1.594536,0.0,0.0


In [10]:
df.to_excel('vivekvihar2025.xlsx', index=False)